#Weather-Aware Trip Planner Assistant

##1: Prepare Colab and Groq access

In [61]:
from google.colab import userdata

api_key = userdata.get('GROQ_API_KEY')
print("Key loaded successfully:", api_key is not None)

Key loaded successfully: True


In [62]:
%pip -q install requests groq langchain-core

In [63]:
import os
from google.colab import userdata
from groq import Groq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
assert os.environ["GROQ_API_KEY"], "Enable GROQ_API_KEY in Colab Secrets."

GROQ_CLIENT = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL_ID = "qwen/qwen3.8-27b"  # Same hosted-Qwen model as the morning activity.

available_ids = {item.id for item in GROQ_CLIENT.models.list().data}
assert MODEL_ID in available_ids, (
    "Selected model unavailable. Check the available tool-capable models."
)
print("Groq client and selected model ready.")

Groq client and selected model ready.


##2: Explore the supplied attractions database

###2.1: Create marathon_db.py

In [64]:
%%writefile marathon_db.py
"""Read-only attractions catalog for the weather and trip-planner marathon."""

import sqlite3
from pathlib import Path

DB_PATH = Path("attractions_sweden_germany.sqlite")

# sample records; coordinates are approximate.
ATTRACTIONS = [
    (1, "Djurgården", "Stockholm", "Sweden", "park", 1, 59.325, 18.103,
     "Large area for outdoor walks and sightseeing."),
      (2, "Skansen Open-Air Museum", "Stockholm", "Sweden", "museum", 1,
     59.324, 18.101, "Open-air cultural attractions and historical buildings."),
    (3, "Vasa Museum", "Stockholm", "Sweden", "museum", 0,
     59.328, 18.091, "Indoor museum with a historic ship."),
    (4, "Slottsskogen", "Gothenburg", "Sweden", "park", 1,
     57.683, 11.947, "Large city park with walking paths."),
    (5, "Gothenburg Botanical Garden", "Gothenburg", "Sweden", "garden", 1,
     57.681, 11.953, "Outdoor botanical garden and walking areas."),
    (6, "Universeum", "Gothenburg", "Sweden", "museum", 0,
     57.695, 11.989, "Indoor science and nature exhibits."),
    (7, "Tiergarten", "Berlin", "Germany", "park", 1,
     52.515, 13.351, "Large green space in central Berlin."),
    (8, "Tempelhofer Feld", "Berlin", "Germany", "park", 1,
     52.473, 13.405, "Open recreational space on a former airfield."),
    (9, "German Museum of Technology", "Berlin", "Germany", "museum", 0,
     52.498, 13.377, "Indoor technology collections."),
    (10, "English Garden", "Munich", "Germany", "park", 1,
     48.152, 11.593, "Outdoor green space and walking routes."),
    (11, "Olympiapark", "Munich", "Germany", "park", 1,
     48.174, 11.549, "Outdoor recreation area and landscaped grounds."),
    (12, "Deutsches Museum", "Munich", "Germany", "museum", 0,
     48.130, 11.583, "Indoor science and technology museum."),
]


def initialize_db():
    with sqlite3.connect(DB_PATH) as connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS attractions (
                id INTEGER PRIMARY KEY,
                name TEXT NOT NULL,
                city TEXT NOT NULL,
                country TEXT NOT NULL,
                category TEXT NOT NULL,
                is_outdoor INTEGER NOT NULL CHECK(is_outdoor IN (0, 1)),
                latitude REAL NOT NULL,
                longitude REAL NOT NULL,
                description TEXT NOT NULL
            )
        """)

        connection.executemany("""
            INSERT OR IGNORE INTO attractions
            (id, name, city, country, category, is_outdoor,
             latitude, longitude, description)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, ATTRACTIONS)

    return len(ATTRACTIONS)


def find_attractions(city, category=None, outdoor_only=True, limit=5):
    """Search the local catalog with parameterized SQL."""

    if not city or not city.strip():
        raise ValueError("city is required")

    if not 1 <= limit <= 12:
        raise ValueError("limit must be from 1 to 12")

    sql = "SELECT * FROM attractions WHERE lower(city) = lower(?)"
    arguments = [city.strip()]

    if outdoor_only:
        # TODO 1A: Restrict results to is_outdoor = 1.
        sql += " AND is_outdoor = ?"
        arguments.append(1)

    if category:
        # TODO 1B: Add a parameterized category filter.
        sql += " AND lower(category) = lower(?)"
        arguments.append(category.strip())

    sql += " ORDER BY name LIMIT ?"
    arguments.append(limit)

    with sqlite3.connect(DB_PATH) as connection:
        connection.row_factory = sqlite3.Row
        rows = connection.execute(sql, arguments).fetchall()

    return [dict(row) for row in rows]


if __name__ == "__main__":
    initialize_db()
    print("Database:", DB_PATH)
    print("Berlin:", find_attractions("Berlin"))

Overwriting marathon_db.py


In [65]:
!python marathon_db.py

Database: attractions_sweden_germany.sqlite
Berlin: [{'id': 8, 'name': 'Tempelhofer Feld', 'city': 'Berlin', 'country': 'Germany', 'category': 'park', 'is_outdoor': 1, 'latitude': 52.473, 'longitude': 13.405, 'description': 'Open recreational space on a former airfield.'}, {'id': 7, 'name': 'Tiergarten', 'city': 'Berlin', 'country': 'Germany', 'category': 'park', 'is_outdoor': 1, 'latitude': 52.515, 'longitude': 13.351, 'description': 'Large green space in central Berlin.'}]


###2.2: TODO 1: Complete the SQL filters

In [66]:
%%writefile marathon_db.py
"""Read-only attractions catalog for the weather and trip-planner marathon."""

import sqlite3
from pathlib import Path

DB_PATH = Path("attractions_sweden_germany.sqlite")

# sample records; coordinates are approximate.
ATTRACTIONS = [
    (1, "Djurgården", "Stockholm", "Sweden", "park", 1, 59.325, 18.103,
     "Large area for outdoor walks and sightseeing."),
      (2, "Skansen Open-Air Museum", "Stockholm", "Sweden", "museum", 1,
     59.324, 18.101, "Open-air cultural attractions and historical buildings."),
    (3, "Vasa Museum", "Stockholm", "Sweden", "museum", 0,
     59.328, 18.091, "Indoor museum with a historic ship."),
    (4, "Slottsskogen", "Gothenburg", "Sweden", "park", 1,
     57.683, 11.947, "Large city park with walking paths."),
    (5, "Gothenburg Botanical Garden", "Gothenburg", "Sweden", "garden", 1,
     57.681, 11.953, "Outdoor botanical garden and walking areas."),
    (6, "Universeum", "Gothenburg", "Sweden", "museum", 0,
     57.695, 11.989, "Indoor science and nature exhibits."),
    (7, "Tiergarten", "Berlin", "Germany", "park", 1,
     52.515, 13.351, "Large green space in central Berlin."),
    (8, "Tempelhofer Feld", "Berlin", "Germany", "park", 1,
     52.473, 13.405, "Open recreational space on a former airfield."),
    (9, "German Museum of Technology", "Berlin", "Germany", "museum", 0,
     52.498, 13.377, "Indoor technology collections."),
    (10, "English Garden", "Munich", "Germany", "park", 1,
     48.152, 11.593, "Outdoor green space and walking routes."),
    (11, "Olympiapark", "Munich", "Germany", "park", 1,
     48.174, 11.549, "Outdoor recreation area and landscaped grounds."),
    (12, "Deutsches Museum", "Munich", "Germany", "museum", 0,
     48.130, 11.583, "Indoor science and technology museum."),
]


def initialize_db():
    with sqlite3.connect(DB_PATH) as connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS attractions (
                id INTEGER PRIMARY KEY,
                name TEXT NOT NULL,
                city TEXT NOT NULL,
                country TEXT NOT NULL,
                category TEXT NOT NULL,
                is_outdoor INTEGER NOT NULL CHECK(is_outdoor IN (0, 1)),
                latitude REAL NOT NULL,
                longitude REAL NOT NULL,
                description TEXT NOT NULL
            )
        """)

        connection.executemany("""
            INSERT OR IGNORE INTO attractions
            (id, name, city, country, category, is_outdoor,
             latitude, longitude, description)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, ATTRACTIONS)

    return len(ATTRACTIONS)


def find_attractions(city, category=None, outdoor_only=True, limit=5):
    """Search the local catalog with parameterized SQL."""

    if not city or not city.strip():
        raise ValueError("city is required")

    if not 1 <= limit <= 12:
        raise ValueError("limit must be from 1 to 12")

    sql = "SELECT * FROM attractions WHERE lower(city) = lower(?)"
    arguments = [city.strip()]

    if outdoor_only:
        sql += " AND is_outdoor = ?"
        arguments.append(1)

    if category:
        sql += " AND lower(category) = lower(?)"
        arguments.append(category.strip())

    sql += " ORDER BY name LIMIT ?"
    arguments.append(limit)

    with sqlite3.connect(DB_PATH) as connection:
        connection.row_factory = sqlite3.Row
        rows = connection.execute(sql, arguments).fetchall()

    return [dict(row) for row in rows]

if __name__ == "__main__":
    initialize_db()
    print("Database:", DB_PATH)
    print("Berlin:", find_attractions("Berlin"))

Overwriting marathon_db.py


In [67]:
!python marathon_db.py

Database: attractions_sweden_germany.sqlite
Berlin: [{'id': 8, 'name': 'Tempelhofer Feld', 'city': 'Berlin', 'country': 'Germany', 'category': 'park', 'is_outdoor': 1, 'latitude': 52.473, 'longitude': 13.405, 'description': 'Open recreational space on a former airfield.'}, {'id': 7, 'name': 'Tiergarten', 'city': 'Berlin', 'country': 'Germany', 'category': 'park', 'is_outdoor': 1, 'latitude': 52.515, 'longitude': 13.351, 'description': 'Large green space in central Berlin.'}]


In [68]:
from marathon_db import initialize_db, find_attractions

initialize_db()

berlin_parks = find_attractions(
    "Berlin",
    category="park",
    outdoor_only=True
)

assert berlin_parks, "Expected outdoor parks in Berlin."
assert all(row["is_outdoor"] == 1 for row in berlin_parks)
assert all(row["category"] == "park" for row in berlin_parks)

stockholm_outdoor = find_attractions(
    "Stockholm",
    outdoor_only=True
)

assert stockholm_outdoor
assert all(row["is_outdoor"] == 1 for row in stockholm_outdoor)

assert find_attractions("Madrid") == []

print("Database filters passed.")

Database filters passed.


##3: Convert the weather API into a forecast function

###3.1: Create marathon_api.py

In [69]:
%%writefile marathon_api.py
"""Open-Meteo daily forecast helper for future trip planning."""
from datetime import datetime, timezone
import requests

FORECAST_URL = "https://api.open-meteo.com/v1/forecast"


def get_forecast(latitude, longitude, days=7):
    """Return compact daily forecasts for a location."""

    if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
        raise ValueError("Invalid latitude or longitude")

    if not 1 <= days <= 7:
        raise ValueError("days must be from 1 to 7")

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": "precipitation_probability_max,wind_speed_10m_max",
        "forecast_days": days,
        "timezone": "auto",
        "wind_speed_unit": "kmh"
    }

    response = requests.get(
        FORECAST_URL,
        params=params,
        timeout=20
    )
    response.raise_for_status()

    data = response.json()
    daily = data["daily"]

    observations = []

    for index, date_string in enumerate(daily["time"]):
        observations.append({
            "date": date_string,
            "rain_probability_max_pct":
                daily["precipitation_probability_max"][index],
            "wind_speed_max_kmh":
                daily["wind_speed_10m_max"][index]
        })

    return {
        "timezone": data.get("timezone"),
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "source": "Open-Meteo Forecast API",
        "days": observations
    }


if __name__ == "__main__":
    sample = get_forecast(52.515, 13.351, days=3)
    print(sample)


Overwriting marathon_api.py


In [70]:

!python marathon_api.py


{'timezone': 'Europe/Berlin', 'retrieved_at_utc': '2026-11-04T13:41:23.384124+00:00', 'source': 'Open-Meteo Forecast API', 'days': [{'date': '2026-11-04', 'rain_probability_max_pct': 0, 'wind_speed_max_kmh': 10.4}, {'date': '2026-11-05', 'rain_probability_max_pct': 0, 'wind_speed_max_kmh': 8.6}, {'date': '2026-11-06', 'rain_probability_max_pct': 0, 'wind_speed_max_kmh': 11.5}]}


###3.2: Test the forecast function

In [71]:

from marathon_api import get_forecast

forecast = get_forecast(52.515, 13.351, days=3)

assert len(forecast["days"]) == 3
assert forecast["timezone"]

for day in forecast["days"]:
    assert isinstance(
        day["rain_probability_max_pct"], (int, float)
    )
    assert isinstance(
        day["wind_speed_max_kmh"], (int, float)
    )
    assert 0 <= day["rain_probability_max_pct"] <= 100
    assert day["wind_speed_max_kmh"] >= 0

print("Forecast extraction passed.")
print(forecast)


Forecast extraction passed.
{'timezone': 'Europe/Berlin', 'retrieved_at_utc': '2026-11-04T13:41:23.588933+00:00', 'source': 'Open-Meteo Forecast API', 'days': [{'date': '2026-11-04', 'rain_probability_max_pct': 0, 'wind_speed_max_kmh': 10.4}, {'date': '2026-11-05', 'rain_probability_max_pct': 0, 'wind_speed_max_kmh': 8.6}, {'date': '2026-11-06', 'rain_probability_max_pct': 0, 'wind_speed_max_kmh': 11.5}]}


##4: Expose the functions as LangChain tools


###4.1: Create marathon_tools.py

In [72]:

%%writefile marathon_tools.py
"""LangChain wrappers around the approved database and API functions."""

from langchain_core.tools import tool
from marathon_db import find_attractions
from marathon_api import get_forecast


@tool
def find_attractions_tool(
    city: str,
    category: str | None = None,
    outdoor_only: bool = True,
    limit: int = 5,
) -> list[dict]:
    """Search seeded attractions in Stockholm, Gothenburg, Berlin, or Munich.

    Use for choosing real candidate places. Returns names, approximate
    coordinates, categories, indoor/outdoor flags, and descriptions.
    Does not provide opening hours or real-time availability.
    """
    return find_attractions(city, category, outdoor_only, limit)


@tool
def get_forecast_tool(
    latitude: float,
    longitude: float,
    days: int = 7
) -> dict:
    """Get a future daily forecast for coordinates from Open-Meteo.

    Returns local dates, daily maximum precipitation probability (%),
    and daily maximum 10-m wind speed (km/h). Call after obtaining
    coordinates from a returned attractions database row.
    """
    return get_forecast(latitude, longitude, days)


APPROVED_TOOLS = [
    find_attractions_tool,
    get_forecast_tool
]

TOOL_REGISTRY = {
    item.name: item for item in APPROVED_TOOLS
}


def groq_schema(item):
    """Convert a LangChain tool into a Groq-compatible function schema."""
    return {
        "type": "function",
        "function": {
            "name": item.name,
            "description": item.description,
            "parameters": item.args_schema.model_json_schema()
        }
    }


TOOL_SCHEMAS = [
    groq_schema(item) for item in APPROVED_TOOLS
]


Overwriting marathon_tools.py


###4.2: Test both tools without an LLM

In [73]:

from marathon_tools import (
    find_attractions_tool,
    get_forecast_tool,
    TOOL_SCHEMAS
)

# Test the database tool
attractions = find_attractions_tool.invoke({
    "city": "Stockholm",
    "outdoor_only": True,
    "limit": 3
})

assert attractions
assert all(place["is_outdoor"] == 1 for place in attractions)

# Use a real returned attraction to test the forecast tool
place = attractions[0]

weather = get_forecast_tool.invoke({
    "latitude": place["latitude"],
    "longitude": place["longitude"],
    "days": 3
})

assert len(weather["days"]) == 3
assert all(
    day["rain_probability_max_pct"] is not None
    for day in weather["days"]
)

print(
    "Registered tool names:",
    [item["function"]["name"] for item in TOOL_SCHEMAS]
)
print("Attraction:", place["name"])
print("Forecast sample:", weather["days"][0])
print("Both tool tests passed.")


Registered tool names: ['find_attractions_tool', 'get_forecast_tool']
Attraction: Djurgården
Forecast sample: {'date': '2026-11-04', 'rain_probability_max_pct': 0, 'wind_speed_max_kmh': 17.3}
Both tool tests passed.


##5: Connect the tools to Groq-hosted Qwen


In [74]:
%%writefile marathon_agent.py
import json
from google.colab import userdata
from groq import Groq

from marathon_tools import TOOL_SCHEMAS, TOOL_REGISTRY

MODEL = "qwen/qwen3.8-27b"

SYSTEM_PROMPT = """
You plan outdoor trips using the attraction database and weather forecast.

Use real attraction records and forecast results. Never invent data.

A suitable day must have BOTH:
- Maximum precipitation probability below 30%.
- Maximum wind speed below 35 km/h.

If no forecast day meets both rules, state that clearly.
Explain the values used. Never claim missing weather data is suitable.
The database does not provide opening hours.
Keep the final answer short.
"""

def run_agent(user_request, max_iterations=5):
    api_key = userdata.get("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("Add GROQ_API_KEY in Colab Secrets.")

    client = Groq(api_key=api_key)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_request},
    ]
    trace = []

    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
            temperature=0.2,
            max_completion_tokens=300,
            reasoning_effort="none",
            parallel_tool_calls=False,
        )

        msg = response.choices[0].message
        calls = msg.tool_calls

        trace.append({
            "iteration": iteration + 1,
            "event": "model_response",
            "content": msg.content,
            "tool_calls": [
                {
                    "id": c.id,
                    "name": c.function.name,
                    "arguments": c.function.arguments,
                }
                for c in (calls or [])
            ],
        })

        if not calls:
            return {
                "answer": msg.content or "",
                "trace": trace,
            }

        messages.append(msg.model_dump(exclude_none=True))

        for call in calls:
            name = call.function.name

            if name not in TOOL_REGISTRY:
                result = {"error": f"Tool not approved: {name}"}
            else:
                try:
                    args = json.loads(call.function.arguments)
                    result = TOOL_REGISTRY[name].invoke(args)
                except Exception as exc:
                    result = {"error": f"{type(exc).__name__}: {exc}"}

            trace.append({
                "iteration": iteration + 1,
                "event": "tool_result",
                "tool": name,
                "result": result,
            })

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": name,
                "content": json.dumps(
                    result, ensure_ascii=False, default=str
                ),
            })

    return {
        "answer": "Tool-call limit reached. Inspect the trace and retry.",
        "trace": trace,
    }


Overwriting marathon_agent.py


In [75]:
import importlib
import marathon_agent

importlib.reload(marathon_agent)

print("Current model:", marathon_agent.MODEL)
print(
    "Current completion limit:",
    marathon_agent.run_agent.__code__.co_consts
)


Current model: qwen/qwen3.8-27b
Current completion limit: (None, 'GROQ_API_KEY', 'Add GROQ_API_KEY in Colab Secrets.', ('api_key',), 'system', ('role', 'content'), 'user', 'auto', 0.2, 300, 'none', False, ('model', 'messages', 'tools', 'tool_choice', 'temperature', 'max_completion_tokens', 'reasoning_effort', 'parallel_tool_calls'), 0, 1, 'model_response', ('id', 'name', 'arguments'), ('iteration', 'event', 'content', 'tool_calls'), '', ('answer', 'trace'), True, ('exclude_none',), 'error', 'Tool not approved: ', ': ', 'tool_result', ('iteration', 'event', 'tool', 'result'), 'tool', ('ensure_ascii', 'default'), ('role', 'tool_call_id', 'name', 'content'), 'Tool-call limit reached. Inspect the trace and retry.')


In [76]:
# Change the model in the actual file.
from pathlib import Path

path = Path("marathon_agent.py")
code = path.read_text(encoding="utf-8")

code = code.replace(
    'MODEL = "qwen/qwen3.8-27b"',
    'MODEL = "llama-3.1-8b-instant"'
)

path.write_text(code, encoding="utf-8")

# Reload the updated module.
import importlib
import marathon_agent

importlib.reload(marathon_agent)

print("Model now set to:", marathon_agent.MODEL)


Model now set to: llama-3.1-8b-instant


In [77]:
from google.colab import userdata
from groq import Groq
from pathlib import Path
import importlib
import marathon_agent

# Check models accessible to your account
client = Groq(api_key=userdata.get("GROQ_API_KEY"))
available = {model.id for model in client.models.list().data}

MODEL = "openai/gpt-oss-20b"

if MODEL not in available:
    print("That model is not available to your account.")
    print("Available models:")
    print("\n".join(sorted(available)))
else:
    path = Path("marathon_agent.py")
    code = path.read_text(encoding="utf-8")

    # Replace whichever model was previously configured.
    import re
    code = re.sub(
        r'^MODEL\s*=\s*["\'].*?["\']',
        f'MODEL = "{MODEL}"',
        code,
        count=1,
        flags=re.MULTILINE,
    )

    # Remove Qwen-specific reasoning parameter, if present.
    code = re.sub(
        r'^\s*reasoning_effort="none",\s*\n',
        '',
        code,
        flags=re.MULTILINE,
    )

    path.write_text(code, encoding="utf-8")
    importlib.reload(marathon_agent)

    print("Agent model:", marathon_agent.MODEL)


Agent model: openai/gpt-oss-20b


In [78]:
result = marathon_agent.run_agent(
    "Find one outdoor attraction in Stockholm and check the forecast. "
    "Recommend a suitable date if both weather rules are met. "
    "Keep your answer brief."
)

print("FINAL RECOMMENDATION")
print(result["answer"])

print("\nTRACE SUMMARY")
for event in result["trace"]:
    print(
        "Iteration:", event["iteration"],
        "| Event:", event["event"],
        "| Tool:", event.get("tool", "")
    )

FINAL RECOMMENDATION
**Outdoor attraction:** Djurgården (park, Stockholm)  
**Suitable date:** 2026‑11‑04  

*Precipitation probability:* 0 % (below 30 %)  
*Maximum wind speed:* 17.3 km/h (below 35 km/h)

TRACE SUMMARY
Iteration: 1 | Event: model_response | Tool: 
Iteration: 1 | Event: tool_result | Tool: find_attractions_tool
Iteration: 2 | Event: model_response | Tool: 
Iteration: 2 | Event: tool_result | Tool: find_attractions_tool
Iteration: 3 | Event: model_response | Tool: 
Iteration: 3 | Event: tool_result | Tool: get_forecast_tool
Iteration: 4 | Event: model_response | Tool: 


In [79]:
print("FINAL ANSWER:")
print(result["answer"])

print("\nTRACE EVENTS:")
for event in result["trace"]:
    print(
        f"Iteration {event['iteration']} | "
        f"{event['event']} | "
        f"{event.get('tool', '')}"
    )

last_event = result["trace"][-1]
print("\nLast event:", last_event["event"])

FINAL ANSWER:
**Outdoor attraction:** Djurgården (park, Stockholm)  
**Suitable date:** 2026‑11‑04  

*Precipitation probability:* 0 % (below 30 %)  
*Maximum wind speed:* 17.3 km/h (below 35 km/h)

TRACE EVENTS:
Iteration 1 | model_response | 
Iteration 1 | tool_result | find_attractions_tool
Iteration 2 | model_response | 
Iteration 2 | tool_result | find_attractions_tool
Iteration 3 | model_response | 
Iteration 3 | tool_result | get_forecast_tool
Iteration 4 | model_response | 

Last event: model_response


In [80]:
import json

called_tools = {
    event["tool"]
    for event in result["trace"]
    if event["event"] == "tool_result"
}

assert "find_attractions_tool" in called_tools
assert "get_forecast_tool" in called_tools
assert result["trace"][-1]["event"] == "model_response"

with open("stockholm_trace.json", "w", encoding="utf-8") as f:
    json.dump(
        result["trace"],
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

print("Tools used:", called_tools)
print("Trace saved to stockholm_trace.json")

Tools used: {'find_attractions_tool', 'get_forecast_tool'}
Trace saved to stockholm_trace.json
